# BharatScore AI v3 — Portfolio Notebook

**Alternative credit scoring for India's credit-invisible borrowers**

| Layer | Model | Algorithm |
|-------|-------|-----------|
| M1 Behavioral | Payment & merchant signals | **XGBoost** |
| M2 Psychometric | Mindset & survey signals | **Random Forest** |
| M3 Liquidity | 6-month cashflow sequences | **LSTM** |
| Meta | Ensemble fusion | **Logistic Regression** |

**Outputs:** Repayment probability · Default probability · Risk band · BharatScore (300–900) · Explainability report

> Production code lives in `data_generator.py`, `feature_engineering.py`, `train.py`, `inference.py`, `app.py`.


In [ ]:
import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    average_precision_score, calibration_curve, classification_report,
    confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
SEED = 42
ARTIFACTS = Path("artifacts")


## 1. Data Generation
Persona-driven synthetic borrowers — correlated features, realistic missingness, **no Faker / no fraud module**.

In [ ]:
from data_generator import BharatDataGenerator, BORROWER_ARCHETYPES

generator = BharatDataGenerator(n_users=6000, seed=SEED)
df_raw, liquidity_sequences = generator.generate()

print(f"Shape: {df_raw.shape} | Sequences: {liquidity_sequences.shape}")
print(f"Default rate: {(1 - df_raw['repaid'].mean()):.1%}")
print(f"Missing cells: {df_raw.isnull().sum().sum()}")
df_raw.head(3)


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df_raw['archetype'].value_counts().plot.bar(ax=ax[0], color="#1A56DB", alpha=0.85)
ax[0].set_title("Borrower Archetypes")
ax[0].tick_params(axis='x', rotation=30)

rates = df_raw.groupby('archetype')['repaid'].apply(lambda s: 1 - s.mean())
rates.plot.bar(ax=ax[1], color="#E02424", alpha=0.85)
ax[1].set_title("Default Rate by Archetype")
ax[1].set_ylabel("Default rate")
plt.tight_layout(); plt.show()


## 2. Feature Engineering
Minimal composite features — strongest signal, minimum complexity.

In [ ]:
from feature_engineering import BharatFeatureEngineer, BEHAVIORAL_FEATURES, PSYCHOMETRIC_FEATURES, LIQUIDITY_TABULAR_FEATURES

df = BharatFeatureEngineer.transform(df_raw)
engineered = [c for c in df.columns if c not in df_raw.columns]
print(f"Added {len(engineered)} engineered features")
df[engineered[:8]].describe().round(3)


## 3. Train Models
Run once via `python train.py`. Notebook loads saved artifacts.

In [ ]:
import train as train_module
config = train_module.train_pipeline(n_users=6000)
config['metrics']


## 4. Evaluation
**ROC-AUC** — rank ordering · **PR-AUC** — imbalanced default detection · **Confusion matrix** — operational errors · **Calibration** — probability trust · **SHAP** — explainability

In [ ]:
eval_npz = np.load(ARTIFACTS / "eval_arrays.npz", allow_pickle=True)
y_test = eval_npz['y_test']
p_ens = eval_npz['p_ens_test']
p1, p2, p3 = eval_npz['p1_test'], eval_npz['p2_test'], eval_npz['p3_test']

print("Ensemble ROC-AUC:", round(roc_auc_score(y_test, p_ens), 4))
print("Ensemble PR-AUC:", round(average_precision_score(y_test, p_ens), 4))
print("\nConfusion Matrix (threshold=0.5):")
print(confusion_matrix(y_test, (p_ens >= 0.5).astype(int)))
print("\n", classification_report(y_test, (p_ens >= 0.5).astype(int), target_names=['Default','Repaid']))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# ROC
for prob, label, color in [(p1,'M1 XGBoost','#1A56DB'), (p2,'M2 RF','#0E9F6E'), (p3,'M3 LSTM','#FF8A4C'), (p_ens,'Ensemble','#111827')]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    axes[0,0].plot(fpr, tpr, label=f"{label} ({roc_auc_score(y_test, prob):.3f})", color=color)
axes[0,0].plot([0,1],[0,1],'k--', alpha=0.4); axes[0,0].legend(); axes[0,0].set_title('ROC Curves')

# PR
prec, rec, _ = precision_recall_curve(y_test, p_ens)
axes[0,1].plot(rec, prec, color='#111827')
axes[0,1].set_title(f'PR Curve — Ensemble (AP={average_precision_score(y_test, p_ens):.3f})')

# Calibration
prob_true, prob_pred = calibration_curve(y_test, p_ens, n_bins=10)
axes[1,0].plot(prob_pred, prob_true, 'o-', label='Ensemble')
axes[1,0].plot([0,1],[0,1],'k--', alpha=0.4)
axes[1,0].set_title('Calibration Plot'); axes[1,0].legend()

# Score distribution
scores = eval_npz['scores']
axes[1,1].hist(scores[y_test==1], bins=30, alpha=0.6, label='Repaid', color='#0E9F6E')
axes[1,1].hist(scores[y_test==0], bins=30, alpha=0.6, label='Default', color='#E02424')
axes[1,1].axvline(550, color='gray', ls='--'); axes[1,1].axvline(750, color='gray', ls='--')
axes[1,1].set_title('BharatScore Distribution'); axes[1,1].legend()
plt.tight_layout(); plt.show()


## 5. SHAP Explainability (M1 Behavioral)

In [ ]:
try:
    import shap
    m1_raw = joblib.load(ARTIFACTS / "m1_xgb_raw.joblib")
    prep_m1 = joblib.load(ARTIFACTS / "prep_m1.joblib")
    from sklearn.model_selection import train_test_split
    X = df.drop(columns=['repaid','archetype','user_id'], errors='ignore')
    X = BharatFeatureEngineer.transform(df_raw)
    _, X_sample = train_test_split(X, train_size=200, random_state=SEED, stratify=df['repaid'])
    X_mat = prep_m1.transform(X_sample)
    explainer = shap.TreeExplainer(m1_raw)
    sv = explainer.shap_values(X_mat)
    if isinstance(sv, list): sv = sv[1]
    shap.summary_plot(sv, X_mat, feature_names=BEHAVIORAL_FEATURES, show=False, max_display=12)
    plt.title('SHAP — M1 Behavioral (XGBoost)'); plt.tight_layout(); plt.show()
except ImportError:
    print("Install shap for SHAP plots: pip install shap")


## 6. Live Inference Demo

In [ ]:
from inference import BharatScoreInference

pipeline = BharatScoreInference()
sample = df_raw.iloc[0].to_dict()
sample['liquidity_sequence'] = liquidity_sequences[0].tolist()
result = pipeline.score(sample)

print(json.dumps(result, indent=2))


## 7. Architecture
```text
Raw Signals → Feature Engineering
    ├─ M1 XGBoost (Behavioral + Merchant)
    ├─ M2 Random Forest (Psychometric)
    └─ M3 LSTM (6×3 monthly cashflow)
         ↓ Meta Logistic Regression [P_M1, P_M2, P_M3]
         ↓ BharatScore 300-900 + Risk Band + SHAP Report
```

**Deploy:** `uvicorn app:app --reload` then `POST /score`